# Testing Models & Data Quality — Exercises

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 5/6
>
> These exercises reinforce the three-layer testing model, pytest mechanics, data validation gates, statistical expectations, and model behaviour tests.

## Exercise 1: Classify Tests into Layers (Conceptual)

You inherit the following test suite. Classify each test into one of the three layers: **Unit**, **Data**, or **Behaviour**.

```python
def test_normalize_age_returns_float():
    assert isinstance(normalize_age(25), float)

def test_batch_has_no_duplicate_ids():
    df = load_latest_batch()
    assert not df['user_id'].duplicated().any()

def test_model_predicts_higher_risk_for_higher_debt():
    pred_low = model.predict([[10000, 50000]])   # debt, income
    pred_high = model.predict([[40000, 50000]])
    assert pred_high > pred_low

def test_feature_pipeline_removes_nulls():
    raw = pd.DataFrame({'age': [25, None, 30]})
    clean = preprocess_features(raw)
    assert clean['age'].isna().sum() == 0

def test_prediction_invariant_to_middle_name():
    pred_a = model.predict([[30, 50000]])  # model sees age, income
    pred_b = model.predict([[30, 50000]])  # middle_name not in features
    assert pred_a == pred_b

def test_incoming_batch_has_required_columns():
    df = load_latest_batch()
    assert set(['user_id', 'age', 'income']).issubset(df.columns)
```

For each test, provide:
1. Its layer (Unit / Data / Behaviour)
2. A one-sentence reason why

In [ ]:
# Your answers here


---

## Exercise 2: Write Basic pytest Tests (Coding)

You have a `calculate_discount` function that applies percentage discounts:

```python
def calculate_discount(price: float, discount_pct: float) -> float:
    """Apply discount_pct (0-100) to price and return final amount."""
    return round(price * (1 - discount_pct / 100), 2)
```

Write **three pytest test functions**:
1. `test_no_discount` — 0% discount returns original price
2. `test_full_discount` — 100% discount returns 0
3. `test_partial_discount` — 20% discount on 100.0 returns 80.0

Run the tests programmatically using `subprocess.run([sys.executable, "-m", "pytest", ...])` and print whether they all passed.

In [ ]:
import subprocess
import sys
from pathlib import Path

# Write calculate_discount function


# Write test file content
test_code = '''
# Your test functions here
'''

# Save test file and run pytest


---

## Exercise 3: Build a Data Validation Gate (Coding)

You receive daily transaction batches. Write a `validate_transactions` function that checks:

- **Required columns**: `transaction_id`, `amount`, `currency`, `timestamp`
- **Null share**: `amount` must have < 1% nulls
- **Range**: `amount` must be between 0.01 and 1,000,000
- **Allowed values**: `currency` must be in {"USD", "EUR", "GBP"}

Return a list of violation messages. An empty list means the batch is valid.

Test your validator on two batches: one clean, one with violations.

In [ ]:
import pandas as pd

def validate_transactions(df: pd.DataFrame) -> list[str]:
    """Validate transaction batch. Returns list of violations (empty = valid)."""
    # Your code here
    pass


# Test with clean batch
clean_batch = pd.DataFrame({
    'transaction_id': [101, 102, 103, 104, 105],
    'amount': [50.0, 120.5, 999.99, 1.50, 340.0],
    'currency': ['USD', 'EUR', 'USD', 'GBP', 'EUR'],
    'timestamp': pd.date_range('2026-08-01', periods=5)
})

# Test with corrupted batch
corrupt_batch = pd.DataFrame({
    'transaction_id': [201, 202, 203],
    'amount': [50.0, None, 2_000_000.0],  # null + out of range
    'currency': ['USD', 'JPY', 'EUR'],    # JPY not allowed
    'timestamp': pd.date_range('2026-08-01', periods=3)
})

# Your test code here


---

## Exercise 4: Statistical Expectations (Coding)

Write a `check_batch_statistics` function that verifies:

1. **ID uniqueness**: No duplicate `customer_id` values
2. **Row count floor**: At least 100 rows present
3. **Mean within band**: Average `purchase_amount` between 20 and 200
4. **Label balance**: `churned` column has both 0 and 1 values (not collapsed to one class)

Return a list of expectation failures. Test on a normal batch and a problematic batch.

In [ ]:
import pandas as pd
import numpy as np

def check_batch_statistics(df: pd.DataFrame) -> list[str]:
    """Check statistical expectations. Returns list of failures."""
    # Your code here
    pass


# Normal batch
np.random.seed(42)
normal_batch = pd.DataFrame({
    'customer_id': range(1, 151),
    'purchase_amount': np.random.uniform(30, 150, 150),
    'churned': np.random.choice([0, 1], 150)
})

# Problematic batch
problem_batch = pd.DataFrame({
    'customer_id': [1, 2, 3, 2, 4],  # duplicate ID
    'purchase_amount': [500, 600, 550, 580, 620],  # mean too high
    'churned': [1, 1, 1, 1, 1]  # collapsed to single class
})

# Your test code here


---

## Exercise 5: Test Model Directionality (Coding)

Train a simple loan approval model (LinearRegression) that predicts approval score based on `income` and `credit_score`. 

Write two behaviour tests:
1. **Direction test**: Higher income (with credit_score fixed) must increase the prediction
2. **Direction test**: Higher credit_score (with income fixed) must increase the prediction

Use assertions that would catch if the model learns the wrong direction.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Train a simple model
X_train = np.array([
    [30000, 600],  # income, credit_score
    [50000, 700],
    [70000, 750],
    [40000, 650],
    [80000, 800]
])
y_train = np.array([50, 70, 85, 60, 95])  # approval scores

model = LinearRegression().fit(X_train, y_train)

# Write your directionality tests here
def test_higher_income_increases_approval():
    pass

def test_higher_credit_score_increases_approval():
    pass

# Run tests


---

## Exercise 6: Test Model Invariance (Coding)

Your student score predictor uses `hours_studied` and `attendance_rate` as features. The API also receives `student_name` but that should NOT affect predictions.

Train a model, then write an invariance test that confirms predictions don't change when only irrelevant metadata differs. The test should compare predictions for the same student features.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# Train model (uses only hours_studied and attendance_rate)
X_train = np.array([
    [2, 0.7],   # hours, attendance
    [5, 0.9],
    [8, 0.95],
    [3, 0.8],
    [6, 0.85]
])
y_train = np.array([45, 68, 88, 55, 72])

model = RandomForestRegressor(random_state=42, n_estimators=10).fit(X_train, y_train)

# Write invariance test
def test_prediction_invariant_to_student_name():
    """Model must ignore student_name since it's not in training features."""
    # Your code here
    pass

# Run test


---

## Exercise 7: Implement a Metric Gate (Coding)

You have a binary classifier. Implement a `deployment_gate` function that checks:

1. **Precision** >= 0.85
2. **Recall** >= 0.80
3. **F1 score** >= 0.82

The function should:
- Accept `y_true`, `y_pred`, and threshold dictionary
- Return `(passed: bool, failures: list[str])`
- A model passes only if ALL gates pass

Test with a model that passes and one that fails.

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def deployment_gate(y_true, y_pred, thresholds: dict) -> tuple[bool, list[str]]:
    """
    Check if model meets deployment thresholds.
    
    Returns:
        (passed, failures): passed=True if all gates pass, failures=list of violations
    """
    # Your code here
    pass


# Test case 1: Good model (should pass)
y_true_good = np.array([1, 0, 1, 1, 0, 1, 0, 1, 1, 0] * 10)
y_pred_good = np.array([1, 0, 1, 1, 0, 1, 0, 1, 0, 0] * 10)  # high precision/recall

# Test case 2: Poor model (should fail)
y_true_poor = np.array([1, 0, 1, 1, 0, 1, 0, 1, 1, 0] * 10)
y_pred_poor = np.array([0, 0, 1, 0, 0, 1, 0, 0, 0, 0] * 10)  # low recall

thresholds = {
    'precision': 0.85,
    'recall': 0.80,
    'f1': 0.82
}

# Your test code here


---

## Exercise 8: When to Use Which Layer? (Conceptual)

For each scenario, identify which test layer(s) would catch the problem and explain why:

1. **Scenario**: Upstream ETL job renamed `user_age` to `age`, breaking your feature pipeline.

2. **Scenario**: A new engineer accidentally swapped the sign in the income coefficient during refactoring.

3. **Scenario**: Vendor changed units from dollars to cents without notice; amounts are now 100x too large.

4. **Scenario**: Your `apply_tax` function returns a string instead of float after a merge conflict.

5. **Scenario**: Training data now has duplicate customer records due to a botched deduplication job.

For each, specify: **Layer** (Unit/Data/Behaviour), **Type of test**, and **Why it catches this failure**.

In [ ]:
# Your answers here


---

## Exercise 9: Build a Complete Validation Pipeline (Challenge)

Create a complete data validation pipeline for a fraud detection system. Implement:

1. **Schema validation**: Required columns, type checks
2. **Statistical expectations**: ID uniqueness, row count floor, mean transaction amount band, fraud label balance
3. **Business rules**: No negative amounts, transaction_date not in future, country codes are valid ISO-2

Your `validate_fraud_batch` function should:
- Take a DataFrame and configuration dict
- Return a structured result: `{"valid": bool, "errors": list, "warnings": list}`
- Distinguish fatal errors (block pipeline) from warnings (log but proceed)

Create three test batches: clean, with warnings only, and with fatal errors.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

VALIDATION_CONFIG = {
    "required_columns": ["transaction_id", "amount", "transaction_date", "country_code", "is_fraud"],
    "id_column": "transaction_id",
    "min_rows": 50,
    "amount_mean_range": (10, 500),
    "valid_country_codes": {"US", "GB", "CA", "AU", "DE", "FR"},
    "label_column": "is_fraud"
}

def validate_fraud_batch(df: pd.DataFrame, config: dict) -> dict:
    """
    Comprehensive validation of fraud detection batch.
    
    Returns:
        {
            "valid": bool,           # False if any fatal errors
            "errors": list[str],     # Fatal issues that block pipeline
            "warnings": list[str]    # Issues to log but allow through
        }
    """
    # Your code here
    pass


# Test batch 1: Clean

# Test batch 2: Warnings only

# Test batch 3: Fatal errors

# Your test code here


---

## Exercise 10: Comprehensive Model Behaviour Suite (Challenge)

Build a complete behaviour test suite for a house price prediction model. The model predicts price based on `square_feet`, `bedrooms`, `bathrooms`, and `age_years`.

Implement tests for:

1. **Directionality** (4 tests):
   - Larger square footage → higher price
   - More bedrooms → higher price
   - More bathrooms → higher price
   - Older age → lower price (newer is better)

2. **Invariance** (1 test):
   - Property address should not affect prediction

3. **Metric gates** (2 tests):
   - R² >= 0.75
   - Mean Absolute Error <= $50,000

4. **Sanity bounds** (1 test):
   - All predictions should be between $50,000 and $2,000,000

Use pytest-style test functions. Run all tests and report results.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Training data
np.random.seed(42)
X_train = np.column_stack([
    np.random.uniform(800, 3500, 100),   # square_feet
    np.random.randint(1, 6, 100),        # bedrooms
    np.random.randint(1, 4, 100),        # bathrooms
    np.random.uniform(0, 50, 100)        # age_years
])

# Price formula: base + sqft*200 + bedrooms*20k + bathrooms*15k - age*1k + noise
y_train = (
    100000 + 
    X_train[:, 0] * 200 + 
    X_train[:, 1] * 20000 + 
    X_train[:, 2] * 15000 - 
    X_train[:, 3] * 1000 + 
    np.random.normal(0, 10000, 100)
)

model = GradientBoostingRegressor(random_state=42, n_estimators=50).fit(X_train, y_train)

# Implement all test functions here

def test_larger_sqft_increases_price():
    pass

# ... more tests ...

# Run all tests and collect results


---

## Exercise 11: Float Comparison Gotchas (Challenge)

The following test fails intermittently due to floating-point precision:

```python
def test_probability_sums_to_one():
    probs = model.predict_proba([[2.5, 0.8]])[0]
    assert probs.sum() == 1.0
```

Tasks:
1. Explain WHY this test is flaky
2. Demonstrate the failure with an example
3. Provide THREE different correct ways to write this test
4. Explain when to use each approach

In [ ]:
# Your solution here


---

## Exercise 12: Design a Testing Strategy (Challenge)

You're building a credit scoring system that:
- Receives daily batches from 3 different data sources
- Merges them into training data
- Trains a weekly model
- Serves predictions via API
- Must comply with fair lending regulations (no discrimination)

Design a complete testing strategy. For each test layer (Unit, Data, Behaviour), specify:

1. **What** you would test
2. **When** the tests run (timing in the pipeline)
3. **What** happens on failure (block, alert, log)
4. **Who** gets notified

Include specific examples of:
- 3 unit tests
- 3 data validation rules per source
- 4 behaviour tests including fairness checks

Present your strategy as a structured document.

### Your Testing Strategy

```
Write your strategy here...
```

---

## 🎯 Bonus Challenge: Catch Real Data Quality Issues

You receive this "clean" dataset from upstream:

```python
customer_data = pd.read_csv('customers.csv')
```

The upstream team swears it's validated. Build validators that would catch:

1. **The ID problem**: IDs look unique in preview but have invisible whitespace differences
2. **The currency problem**: Amounts are mixed USD and cents (some 100x off)
3. **The timezone problem**: Timestamps from different sources have different timezones
4. **The encoding problem**: Names with unicode characters are corrupted
5. **The silent null problem**: Nulls encoded as string "NULL", "N/A", "missing"

For each, write a validator that catches it and explain how you'd fix it in production.

In [ ]:
# Create synthetic datasets demonstrating each problem
# Then write validators that catch them
